# Text Preprocessing — From Raw Text to Numbers

Computers don't understand words. They understand numbers. Every NLP system starts with the same
challenge: how do we turn messy, ambiguous human language into something a machine can work with?

This notebook covers the **traditional NLP pipeline** — the foundation everything else builds on.

---
## Why Text Is Hard for Computers

Images are grids of pixel values. Tabular data has rows and columns. Text is... a mess.

| Challenge | Example |
|---|---|
| **Unstructured** | No fixed format — tweets, essays, legal docs all look different |
| **Variable length** | "Hi" vs a 10,000-word article — how do you feed both into a model? |
| **Ambiguous** | "I saw her duck" — did she duck, or did I see her pet duck? |
| **Context-dependent** | "It's cold" means different things in a weather report vs a thriller novel |
| **Synonyms & polysemy** | "car" = "automobile"; "bank" = river bank OR financial bank |

The NLP pipeline exists to tame this chaos, one step at a time.

In [ ]:
sample_text = """
The movie was absolutely TERRIBLE! I can't believe I wasted $15 on this.
The acting was bad, the plot didn't make sense, and it was way too long.
Do NOT watch this movie. 0/10 would not recommend.
""".strip()

print(sample_text)
print(f"\nLength: {len(sample_text)} characters")
print(f"A model sees: {[ord(c) for c in sample_text[:30]]}...")

---
## Tokenization

The first step: split text into meaningful pieces called **tokens**.

- **Word tokenization** — split on spaces/punctuation → `["The", "movie", "was", ...]`
- **Sentence tokenization** — split into sentences
- **Subword tokenization** — split into pieces smaller than words (used by modern models like BERT/GPT)

NLTK handles the common cases well.

In [ ]:
import nltk
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

from nltk.tokenize import word_tokenize, sent_tokenize

sentences = sent_tokenize(sample_text)
print("Sentences:")
for i, s in enumerate(sentences):
    print(f"  {i+1}. {s}")

print(f"\nWord tokens:")
tokens = word_tokenize(sample_text)
print(tokens)
print(f"\n{len(tokens)} tokens from {len(sample_text)} characters")

In [ ]:
naive_split = sample_text.split()
print("Naive split:", naive_split[:8])
print("NLTK split:", tokens[:8])
print("\nNaive keeps punctuation attached: 'TERRIBLE!' vs NLTK separates: 'TERRIBLE', '!'")

---
## Lowercasing, Punctuation Removal, Stopwords

Raw tokens are noisy. We clean them up:

1. **Lowercase** — "Movie" and "movie" should be the same token
2. **Remove punctuation** — "!" and "." don't carry meaning for most tasks
3. **Remove stopwords** — ultra-common words ("the", "is", "a") that add noise without meaning

In [ ]:
from nltk.corpus import stopwords
import string

stop_words = set(stopwords.words('english'))

tokens_lower = [t.lower() for t in tokens]
tokens_no_punct = [t for t in tokens_lower if t not in string.punctuation and t.isalpha()]
tokens_no_stop = [t for t in tokens_no_punct if t not in stop_words]

print(f"Original ({len(tokens)}):  {tokens[:10]}")
print(f"Lowered  ({len(tokens_lower)}):  {tokens_lower[:10]}")
print(f"No punct ({len(tokens_no_punct)}): {tokens_no_punct[:10]}")
print(f"No stops ({len(tokens_no_stop)}): {tokens_no_stop}")

In [ ]:
print("Stopwords removed:", [t for t in tokens_no_punct if t in stop_words])
print(f"\nWe dropped {len(tokens_no_punct) - len(tokens_no_stop)} stopwords — nearly half the tokens.")
print("What remains captures the actual meaning: terrible, believe, wasted, bad, plot, ...")

---
## Stemming vs Lemmatization

Different forms of the same word should map to one representation:

| Word | Stem | Lemma |
|---|---|---|
| running | runn | run |
| better | better | good |
| studies | studi | study |

- **Stemming** — crude chopping (fast, often produces non-words)
- **Lemmatization** — dictionary lookup (slower, produces real words)

In [ ]:
from nltk.stem import PorterStemmer, WordNetLemmatizer

stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

test_words = ["running", "better", "studies", "happily", "connection", "was", "mice", "geese"]

print(f"{'Word':<15} {'Stem':<15} {'Lemma':<15}")
print("-" * 45)
for w in test_words:
    print(f"{w:<15} {stemmer.stem(w):<15} {lemmatizer.lemmatize(w, pos='v'):<15}")

---
## Bag of Words (BoW)

The simplest way to turn text into a vector: **count how many times each word appears**.

```
"I love NLP"         → [1, 1, 1, 0, 0]
"I love deep NLP"    → [1, 1, 1, 1, 0]
"I hate Mondays"     → [1, 0, 0, 0, 1]
                        I  love NLP deep hate ...
```

Each document becomes a vector over the entire vocabulary. Most entries are zero (sparse).

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd

corpus = [
    "I love this movie it was great",
    "I hate this movie it was terrible",
    "The movie was okay not great not terrible",
    "Absolutely loved it best movie ever",
]

vectorizer = CountVectorizer()
bow_matrix = vectorizer.fit_transform(corpus)

df = pd.DataFrame(bow_matrix.toarray(), columns=vectorizer.get_feature_names_out())
df.index = [f"doc_{i}" for i in range(len(corpus))]
print(f"Vocabulary size: {len(vectorizer.get_feature_names_out())}")
print(f"Matrix shape: {bow_matrix.shape} (docs × words)")
print(f"Non-zero entries: {bow_matrix.nnz} out of {bow_matrix.shape[0] * bow_matrix.shape[1]}\n")
df

---
## TF-IDF — Words Weighted by Importance

BoW treats every word equally. But "the" appearing 10 times isn't 10× more informative.

**TF-IDF** = Term Frequency × Inverse Document Frequency

- **TF**: how often the word appears in THIS document (more = more relevant to this doc)
- **IDF**: how rare the word is ACROSS ALL documents (rarer = more distinctive)

$$\text{TF-IDF}(t, d) = \text{TF}(t, d) \times \log\left(\frac{N}{\text{DF}(t)}\right)$$

Words like "the" get low scores (common everywhere). Words like "terrible" get high scores (distinctive).

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(corpus)

df_tfidf = pd.DataFrame(
    tfidf_matrix.toarray().round(2),
    columns=tfidf.get_feature_names_out()
)
df_tfidf.index = [f"doc_{i}" for i in range(len(corpus))]
df_tfidf

In [ ]:
print("Top TF-IDF words per document:\n")
for i, doc in enumerate(corpus):
    scores = dict(zip(tfidf.get_feature_names_out(), tfidf_matrix[i].toarray()[0]))
    top = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:3]
    print(f"doc_{i}: {doc[:40]}...")
    print(f"  Top words: {[(w, round(s, 3)) for w, s in top]}\n")

---
## Build a Text Classifier — TF-IDF + Logistic Regression

Let's put it all together: take raw text, vectorize with TF-IDF, and train a classifier.
This simple pipeline is still surprisingly effective for many real-world tasks.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

reviews = [
    "This movie was fantastic and amazing",
    "Loved every minute of this film",
    "Great acting and wonderful story",
    "Best movie I have seen this year",
    "Absolutely brilliant and heartwarming",
    "A masterpiece of modern cinema",
    "Incredible performances by the entire cast",
    "Beautiful cinematography and moving soundtrack",
    "This movie was terrible and boring",
    "Hated it worst film ever made",
    "Awful acting and no plot whatsoever",
    "Waste of time do not watch",
    "Horrible movie with bad dialogue",
    "Painfully slow and completely pointless",
    "The worst movie I have ever seen",
    "Dreadful story and terrible effects",
]
labels = [1]*8 + [0]*8  # 1=positive, 0=negative

X_train, X_test, y_train, y_test = train_test_split(
    reviews, labels, test_size=0.25, random_state=42
)

tfidf_clf = TfidfVectorizer(ngram_range=(1, 2))
X_train_vec = tfidf_clf.fit_transform(X_train)
X_test_vec = tfidf_clf.transform(X_test)

model = LogisticRegression()
model.fit(X_train_vec, y_train)

y_pred = model.predict(X_test_vec)
print(classification_report(y_test, y_pred, target_names=["negative", "positive"]))

In [ ]:
new_reviews = [
    "This was a great film I enjoyed it",
    "Terrible movie worst I have seen",
    "The acting was okay but the story was weak",
]

new_vec = tfidf_clf.transform(new_reviews)
predictions = model.predict(new_vec)
probs = model.predict_proba(new_vec)

for review, pred, prob in zip(new_reviews, predictions, probs):
    sentiment = "positive" if pred == 1 else "negative"
    confidence = max(prob) * 100
    print(f"{sentiment:>8} ({confidence:.0f}%) | {review}")

---
## N-grams — Capturing Word Pairs and Triples

Unigrams (single words) miss multi-word expressions:

- "not good" → unigrams see "not" and "good" separately
- "New York" → unigrams see "New" and "York" separately

**N-grams** capture sequences: bigrams (2 words), trigrams (3 words).

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

texts = [
    "The movie was not good",
    "The movie was good",
]

uni = CountVectorizer(ngram_range=(1, 1))
bi = CountVectorizer(ngram_range=(1, 2))

uni_matrix = uni.fit_transform(texts)
bi_matrix = bi.fit_transform(texts)

print("UNIGRAMS:")
print(f"  Vocabulary: {uni.get_feature_names_out().tolist()}")
print(f"  'not good' vector: {uni_matrix[0].toarray()[0]}")
print(f"  'good' vector:     {uni_matrix[1].toarray()[0]}")

from sklearn.metrics.pairwise import cosine_similarity
sim = cosine_similarity(uni_matrix[0], uni_matrix[1])[0][0]
print(f"  Cosine similarity: {sim:.3f} — they look almost identical!\n")

print("BIGRAMS:")
print(f"  Vocabulary: {bi.get_feature_names_out().tolist()}")
print(f"  'not good' vector: {bi_matrix[0].toarray()[0]}")
print(f"  'good' vector:     {bi_matrix[1].toarray()[0]}")

sim_bi = cosine_similarity(bi_matrix[0], bi_matrix[1])[0][0]
print(f"  Cosine similarity: {sim_bi:.3f} — now they're more different!")

---
## Limitations of BoW and TF-IDF

These methods got us far, but they have fundamental problems:

| Limitation | Example |
|---|---|
| **No word order** | "Dog bites man" ≈ "Man bites dog" (same BoW vector!) |
| **No semantics** | "happy" and "joyful" are treated as completely unrelated |
| **Sparse & high-dimensional** | 50K+ vocabulary → 50K-dimensional sparse vectors |
| **No context** | "bank" always gets the same weight regardless of river or money |

We need representations that understand **meaning**, not just counts.

→ **Next: Word Embeddings** — dense vectors where similar words are close together.

In [ ]:
demo_texts = [
    "The dog bites the man",
    "The man bites the dog",
    "I am happy and joyful",
    "I am glad and cheerful",
]

vec = TfidfVectorizer()
vecs = vec.fit_transform(demo_texts)

print("Cosine similarities (TF-IDF):")
print(f"  'dog bites man' vs 'man bites dog': {cosine_similarity(vecs[0], vecs[1])[0][0]:.3f}  ← should be DIFFERENT meanings!")
print(f"  'happy joyful' vs 'glad cheerful':  {cosine_similarity(vecs[2], vecs[3])[0][0]:.3f}  ← should be SIMILAR meanings!")
print("\nTF-IDF can't capture word order or synonyms. We need something better.")

---

## Summary

The traditional NLP pipeline:

```
Raw Text → Tokenize → Clean (lower, stopwords, stem/lemma) → Vectorize (BoW/TF-IDF) → ML Model
```

**Key takeaways:**
- Tokenization breaks text into processable units
- Cleaning removes noise (stopwords, punctuation, case differences)
- BoW counts words; TF-IDF weights them by importance
- N-grams capture multi-word patterns
- These methods work well for classification but fundamentally lose meaning and order

**Next notebook:** Word embeddings solve the "meaning" problem by placing words in a semantic vector space.